# trl-for-angels: Kaggle runner

The only notebook you need. Each run it pulls the project from GitHub, runs the task in
`kaggle/run.sh` (Claude picks it), and pushes the results to the `kaggle-results` branch,
where Claude reads them. You don't need to watch it.

**One-time setup**
1. Settings → **Accelerator: GPU T4** and **Internet: On**.
2. Add-ons → **Secrets**, attached to this notebook:
   - `GITHUB_TOKEN`: a fine-grained GitHub token for `Sidi3355/trl-for-angels` only, with
     **Contents: Read and write**.
   - `ANTHROPIC_API_KEY`: your Claude Console key (needed from step A4).

**Each run:** **Save Version → Save & Run All (Commit) → Save**. It runs in the background
on Kaggle; close the tab if you like. Then tell Claude it's started.

In [ ]:
# Secrets from Add-ons > Secrets, into this kernel's environment (never printed).
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
for name in ("GITHUB_TOKEN", "ANTHROPIC_API_KEY"):
    try:
        os.environ[name] = secrets.get_secret(name)
        print(f"{name}: attached")
    except Exception:
        print(f"{name}: not attached")
assert os.environ.get("GITHUB_TOKEN"), "Attach the GITHUB_TOKEN secret (Add-ons > Secrets)"

In [ ]:
%%bash
# This project's repo, at the branch Claude works on.
set -euo pipefail
AUTH="AUTHORIZATION: basic $(printf 'x-access-token:%s' "$GITHUB_TOKEN" | base64 -w0)"
mkdir -p /tmp/work && cd /tmp/work && rm -rf repo
git -c http.extraHeader="$AUTH" clone -q --depth 1 -b claude/determined-albattani-dl2cyy https://github.com/Sidi3355/trl-for-angels.git repo
git -C repo log --oneline -1

In [ ]:
%%bash
# Installs Kev (pinned) and our TRL fork into their own Python 3.13 environments under /tmp/work.
set -euo pipefail
# Kaggle's PYTHONPATH carries a sitecustomize hook that needs packages our venvs don't have.
unset PYTHONPATH
nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv \
  || { echo "NO GPU: set Settings > Accelerator to GPU T4, then restart"; exit 1; }
pip install -q uv
mkdir -p /tmp/work && cd /tmp/work
[ -d kev ] || git clone -q https://github.com/jaredpalmer/kev.git
git -C kev checkout -q 2074cc4
(cd kev && python -m uv sync -q --extra serve)
[ -d think-reason-learn ] || git clone -q -b feature/decision-combiner https://github.com/Sidi3355/think-reason-learn.git
[ -d trl-venv ] || python -m uv venv -q --python 3.13 trl-venv
python -m uv pip install -q --python trl-venv/bin/python -e "think-reason-learn[anthropic]"
trl-venv/bin/python -c "import think_reason_learn as t; print('TRL', t.__version__)"
kev/.venv/bin/python -c "import torch; print('torch', torch.__version__, 'CUDA available:', torch.cuda.is_available())"

In [ ]:
%%bash
# Runs whatever kaggle/run.sh says. Never fails the cell, so the results still get pushed.
unset PYTHONPATH
export REPO=/tmp/work/repo WORK=/tmp/work OUT=/kaggle/working/out
rm -rf "$OUT" && mkdir -p "$OUT"
bash "$REPO/kaggle/run.sh" 2>&1 | tee "$OUT/run.log"
echo "run.sh exit: ${PIPESTATUS[0]}" | tee -a "$OUT/run.log"

In [ ]:
%%bash
# Pushes this run's small files (no model weights) to the kaggle-results branch.
set -euo pipefail
AUTH="AUTHORIZATION: basic $(printf 'x-access-token:%s' "$GITHUB_TOKEN" | base64 -w0)"
OUT=/kaggle/working/out
STAMP=$(date -u +%Y%m%d-%H%M%S)
TASK=$(cat "$OUT/task.txt" 2>/dev/null || echo unknown)
cd /tmp/work && rm -rf results
if ! git -c http.extraHeader="$AUTH" clone -q --depth 1 -b kaggle-results https://github.com/Sidi3355/trl-for-angels.git results 2>/dev/null; then
  git -c http.extraHeader="$AUTH" clone -q --depth 1 https://github.com/Sidi3355/trl-for-angels.git results
  (cd results && git checkout -q --orphan kaggle-results && git rm -rqf --ignore-unmatch .)
fi
DEST="/tmp/work/results/runs/$STAMP-$TASK"
mkdir -p "$DEST"
(cd "$OUT" && find . -type f -size -5M ! -name '*.safetensors' ! -name '*.pt' ! -name '*.bin' \
   -exec cp --parents {} "$DEST/" \;)
cd results
git config user.name "kaggle-runner"
git config user.email "kaggle-runner@users.noreply.github.com"
git add -A && git commit -q -m "Kaggle run $STAMP ($TASK)"
git -c http.extraHeader="$AUTH" push -q origin kaggle-results
echo "Pushed runs/$STAMP-$TASK to the kaggle-results branch."